### Basic neural activity analysis with single camera tracking
#### use GLM model to analyze spike count trains, the GLM use continuous variables and use basis kernel to simplify the fitting
#### also add the function to reduce the continuous variables into smaller dimensions to reduce the correlation and be more task related 
#### add the self force (delta force) as one variable in the model

In [ ]:
import pandas as pd
import numpy as np
from numpy import genfromtxt
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn
import scipy
import scipy.stats as st
import scipy.io
from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA
import string
import warnings
import pickle
import json

import statsmodels.api as sm

import os
import glob
import random
from time import time

from scipy.ndimage import label


### function - get body part location for each pair of cameras

In [ ]:
from ana_functions.body_part_locs_eachpair import body_part_locs_eachpair
from ana_functions.body_part_locs_singlecam import body_part_locs_singlecam

### function - align the two cameras

In [ ]:
from ana_functions.camera_align import camera_align       

### function - merge the two pairs of cameras

In [ ]:
from ana_functions.camera_merge import camera_merge

### function - find social gaze time point

In [ ]:
from ana_functions.find_socialgaze_timepoint import find_socialgaze_timepoint
from ana_functions.find_socialgaze_timepoint_singlecam import find_socialgaze_timepoint_singlecam
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody import find_socialgaze_timepoint_singlecam_wholebody
from ana_functions.find_socialgaze_timepoint_singlecam_wholebody_2 import find_socialgaze_timepoint_singlecam_wholebody_2




### function - define time point of behavioral events

In [ ]:
from ana_functions.bhv_events_timepoint import bhv_events_timepoint
from ana_functions.bhv_events_timepoint_singlecam import bhv_events_timepoint_singlecam

### function - plot behavioral events

In [ ]:
from ana_functions.plot_bhv_events import plot_bhv_events
from ana_functions.plot_bhv_events_levertube import plot_bhv_events_levertube
from ana_functions.draw_self_loop import draw_self_loop
import matplotlib.patches as mpatches 
from matplotlib.collections import PatchCollection

### function - plot inter-pull interval

In [ ]:
from ana_functions.plot_interpull_interval import plot_interpull_interval

### function - interval between all behavioral events

In [ ]:
from ana_functions.bhv_events_interval import bhv_events_interval

### function - GLM fitting for spike trains based on the discrete variables from single camera

In [ ]:
from ana_functions.singlecam_bhv_var_neuralGLM_fitting_BasisKernelsForContVaris import get_singlecam_bhv_var_for_neuralGLM_fitting_BasisKernelsForContVaris
from ana_functions.singlecam_bhv_var_neuralGLM_fitting_BasisKernelsForContVaris import neuralGLM_fitting_BasisKernelsForContVaris

from ana_functions.singlecam_bhv_var_neuralGLM_fitting_BasisKernelsForContVaris_PullGazeVectorProjection \
import neuralGLM_fitting_BasisKernelsForContVaris_PullGazeVectorProjection

from ana_functions.singlecam_bhv_var_neuralGLM_fitting_BasisKernelsForContVaris_PullGazeVectorProjection \
import neuralGLM_fitting_BasisKernelsForContVaris_PullGazeVectorProjection_partnerPC1

from ana_functions.singlecam_bhv_var_neuralGLM_fitting_BasisKernelsForContVaris_PullGazeVectorProjection \
import neuralGLM_fitting_BasisKernelsForContVaris_PullGazeAxis_partnerPC1

from ana_functions.singlecam_bhv_var_neuralGLM_fitting_BasisKernelsForContVaris_PullGazeVectorProjection \
import neuralGLM_fitting_BasisKernelsForContVaris_PullGazeAxis_partnerPullGazeAxis

from ana_functions.singlecam_bhv_var_neuralGLM_fitting_BasisKernelsForContVaris_PullGazeVectorProjection \
import neuralGLM_fitting_BasisKernelsForContVaris_PullGazeAxis_partnerPC1_LOOmethods


### function - other useful functions

In [ ]:
# for defining the meaningful social gaze (the continuous gaze distribution that is closest to the pull) 
from ana_functions.keep_closest_cluster_single_trial import keep_closest_cluster_single_trial

In [ ]:
# get useful information about pulls
from ana_functions.get_pull_infos import get_pull_infos

In [ ]:
# use the gaze vector speed and face mass speed to find the pull action start time within IPI
from ana_functions.find_sharp_increases_withinIPI import find_sharp_increases_withinIPI
from ana_functions.find_sharp_increases_withinIPI import find_sharp_increases_withinIPI_dual_speed

In [ ]:
from ana_functions.cluster_based_correction_with_timing import cluster_based_correction_with_timing

In [ ]:
# check the orthogonality among the three behavioral vectors
from ana_functions.check_orthogonality import check_orthogonality
from ana_functions.check_orthogonality import gram_schmidt

## Analyze each session

### prepare the basic behavioral data (especially the time stamps for each bhv events)

In [ ]:
# instead of using gaze angle threshold, use the target rectagon to deside gaze info
# ...need to update
sqr_thres_tubelever = 75 # draw the square around tube and lever
sqr_thres_face = 1.15 # a ratio for defining face boundary
sqr_thres_body = 4 # how many times to enlongate the face box boundry to the body


# get the fps of the analyzed video
fps = 30

# get the fs for neural recording
fs_spikes = 20000
fs_lfp = 1000

# frame number of the demo video
# nframes = 0.5*30 # second*30fps
nframes = 45*30 # second*30fps

# re-analyze the video or not
reanalyze_video = 0
redo_anystep = 0

# force manipulation type
# SR_bothchange: self reward, both forces changed
# CO_bothchange: 1s cooperation, both forces changed
# CO_A1change: 1s cooperation, animal 1 forces changed
# CO_A2change: 1s cooperation, animal 2 forces changed

# do OFC sessions or DLPFC sessions
do_OFC = 0
do_DLPFC  = 1
if do_OFC:
    savefile_sufix = '_OFCs'
elif do_DLPFC:
    savefile_sufix = '_DLPFCs'
else:
    savefile_sufix = ''
    
    
# glm types
# glm_encoding_type = 'withpartnerPC1_' # '', 'withpartnerPC1_',  'pullgazeAxis_partnerPC1_', 'orinigalV1to8_'(do not use)
glm_encoding_type = 'pullgazeAxis_partnerPC1_' # '', 'withpartnerPC1_',  'pullgazeAxis_partnerPC1_', 'orinigalV1to8_'(do not use)
# glm_encoding_type = '' # empty means uce the projected 'PCs' to the pull and gaze axis, but did not consider partnerPC1
    

# dannon kanga 
if 1:
    if do_DLPFC:
        neural_record_conditions = [
                                    '20240910_Kanga_EffortBasedMC',       '20240911_Kanga_EffortBasedMC',
                                    '20240912_Kanga_EffortBasedSR',       '20240913_Kanga_EffortBasedSR',
                                    '20240916_Kanga_EffortBasedMC',       '20240917_Kanga_EffortBasedSR',
                                    '20240918_Kanga_EffortBasedMC',       '20241008_Kanga_EffortBasedMC',
                                    '20241009_Kanga_DannonEffortBasedMC', '20241010_Kanga_EffortBasedMC',
                                    '20241011_Kanga_DannonEffortBasedMC', '20241014_Kanga_EffortBasedMC',
                                    '20241016_Kanga_DannonEffortBasedMC', '20241017_Kanga_EffortBasedMC',
                                    '20241018_Kanga_DannonEffortBasedMC', '20241022_Kanga_DannonEffortBasedMC',
                                    '20241025_Kanga_DannonEffortBasedMC', '20241101_Kanga_EffortBasedSR',
                                    '20241104_Kanga_EffortBasedSR',
                                   ]
        # self and other are corresponding to the recorded animals
        task_conditions = [
                            'self_EffortBasedMC',  'self_EffortBasedMC',
                            'self_EffortBasedSR',  'self_EffortBasedSR',
                            'self_EffortBasedMC',  'self_EffortBasedSR',
                            'self_EffortBasedMC',  'self_EffortBasedMC',
                            'other_EffortBasedMC', 'self_EffortBasedMC',
                            'other_EffortBasedMC', 'self_EffortBasedMC',
                            'other_EffortBasedMC', 'self_EffortBasedMC',
                            'other_EffortBasedMC', 'other_EffortBasedMC',
                            'other_EffortBasedMC', 'self_EffortBasedSR',
                            'self_EffortBasedSR',
                          ]
        dates_list = [
                        '20240910', '20240911', '20240912', '20240913',
                        '20240916', '20240917', '20240918', '20241008',
                        '20241009', '20241010', '20241011', '20241014',
                        '20241016', '20241017', '20241018', '20241022',
                        '20241025', '20241101', '20241104',
                     ]
        videodates_list = [
                            '20240910', '20240911', '20240912', '20240913',
                            '20240916', '20240917', '20240918', '20241008',
                            '20241009', '20241010', '20241011', '20241014',
                            '20241016', '20241017', '20241018', '20241022',
                            '20241025', '20241101', '20241104',
                          ] # to deal with the sessions that MC and SR were in the same session
        session_start_times = [ 
                                0.00, 0.00, 90.1, 69.5,
                                62.5, 0.00, 43.5, 59.6,
                                0.00, 66.0, 0.00, 0.00,
                                0.00, 0.00, 0.00, 0.00,
                                0.00, 0.00, 0.00,
                              ] # in second
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0']*np.shape(dates_list)[0]
        
        animal1_fixedorders = ['dannon']*np.shape(dates_list)[0]
        animal2_fixedorders = ['kanga']*np.shape(dates_list)[0]

        animal1_filenames = ["Dannon"]*np.shape(dates_list)[0]
        animal2_filenames = ["Kanga"]*np.shape(dates_list)[0]
        
        recordedanimals = animal2_fixedorders 
        
    elif do_OFC:
        neural_record_conditions = [
                                
                                   ]
        task_conditions = [
                            
                          ]
        dates_list = [
                      
                     ]
        videodates_list = dates_list
        session_start_times = [ 
                                
                              ] # in second
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0']*np.shape(dates_list)[0]
    
        animal1_fixedorders = ['dannon']*np.shape(dates_list)[0]
        animal2_fixedorders = ['kanga']*np.shape(dates_list)[0]

        animal1_filenames = ["Dannon"]*np.shape(dates_list)[0]
        animal2_filenames = ["Kanga"]*np.shape(dates_list)[0]
        
        recordedanimals = animal2_fixedorders 
        
        
# dodson kanga (kanga recorded) - do not use becuase the data is not great. consider to conbine with previous section later (all kanga together)
if 0:
    if do_DLPFC:
        neural_record_conditions = [
                        '20250505_Kanga_EffortBasedSR_withDodson', '20250506_Kanga_EffortBasedSR_withDodson', 
                        '20250508_Kanga_EffortBasedSR_withDodson', '20250516_Kanga_EffortBasedSR_withDodson', 
                        '20250519_Kanga_EffortBasedSR_withDodson', '20250520_Kanga_EffortBasedSR_withDodson', 
                        '20250522_Kanga_EffortBasedSR_withDodson', '20250523_Kanga_EffortBasedSR_withDodson', 
            
                        '20250506_Kanga_DodsonEffortBasedMC',      '20250508_Kanga_DodsonEffortBasedMC',  
                        '20250512_Kanga_DodsonEffortBasedMC',      '20250520_Kanga_DodsonEffortBasedMC',  
                        '20250523_Kanga_DodsonEffortBasedMC', 
                        
                        '20250507_Kanga_KangaEffortBasedMC',       '20250516_Kanga_KangaEffortBasedMC',   
                        '20250522_Kanga_KangaEffortBasedMC',   
                                   ]
        # self and other are corresponding to the recorded animals
        task_conditions = [
                            'self_EffortBasedSR', 'self_EffortBasedSR',
                            'self_EffortBasedSR', 'self_EffortBasedSR',
                            'self_EffortBasedSR', 'self_EffortBasedSR',
                            'self_EffortBasedSR', 'self_EffortBasedSR',
                            
                            'other_EffortBasedMC', 'other_EffortBasedMC',
                            'other_EffortBasedMC', 'other_EffortBasedMC',
                            'other_EffortBasedMC', 
            
            
                            'self_EffortBasedMC',  'self_EffortBasedMC',
                            'self_EffortBasedMC',  
                          ]
        dates_list = [
                        # both animals' lever force were changed - Self reward
                        "20250505",    "20250506_SR", "20250508_SR", "20250516_SR",
                        "20250519",    "20250520_SR", "20250522_SR", "20250523_SR",
                        # Dodson's lever force were changed
                        "20250506_MC", "20250508",    "20250512_MC", "20250520",
                        "20250523", 
                        # Kanga's lever force were changed
                        "20250507",    "20250516_MC", "20250522"
                     ]
        videodates_list = [
                            # "both animals' lever force were changed - Self reward
                            "20250505",    "20250506_SR", "20250508_SR", "20250516_SR",
                            "20250519",    "20250520_SR", "20250522_SR", "20250523_SR",
                            # Dodson's lever force were changed
                            "20250506_MC", "20250508",    "20250512_MC", "20250520",
                            "20250523", 
                            # Kanga's lever force were changed
                            "20250507",    "20250516_MC", "20250522"
                          ] # to deal with the sessions that MC and SR were in the same session
        session_start_times = [ 
                                147.1, 0.00, 0.00, 0.00,
                                106.0, 0.00, 0.00, 257.0,    
                                
                                0.00, 151.0, 265.0, 0.00, 
                                66.4,
            
                                0.00, 171.1, 0.00,                     
                              ] # in second
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai9']*np.shape(dates_list)[0]
        
        animal1_fixedorders = ['dodson']*np.shape(dates_list)[0]
        animal2_fixedorders = ['kanga']*np.shape(dates_list)[0]

        animal1_filenames = ["Dodson"]*np.shape(dates_list)[0]
        animal2_filenames = ["Kanga"]*np.shape(dates_list)[0]
        
        recordedanimals = animal2_fixedorders 
        
    elif do_OFC:
        neural_record_conditions = [
                                
                                   ]
        task_conditions = [
                            
                          ]
        dates_list = [
                      
                     ]
        videodates_list = dates_list
        session_start_times = [ 
                                
                              ] # in second
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai9']*np.shape(dates_list)[0]
    
        animal1_fixedorders = ['dodson']*np.shape(dates_list)[0]
        animal2_fixedorders = ['kanga']*np.shape(dates_list)[0]

        animal1_filenames = ["Dodson"]*np.shape(dates_list)[0]
        animal2_filenames = ["Kanga"]*np.shape(dates_list)[0]
        
        recordedanimals = animal2_fixedorders 
        
        
# dodson kanga (dodson recorded)
if 0:
    if do_DLPFC:
        neural_record_conditions = [
                        '20250505_Dodson_EffortBasedSR_withKanga', '20250506_Dodson_EffortBasedSR_withKanga', 
                        '20250508_Dodson_EffortBasedSR_withKanga', '20250516_Dodson_EffortBasedSR_withKanga', 
                        '20250519_Dodson_EffortBasedSR_withKanga', '20250520_Dodson_EffortBasedSR_withKanga', 
                        '20250522_Dodson_EffortBasedSR_withKanga', '20250523_Dodson_EffortBasedSR_withKanga', 
            
                        '20250506_Dodson_DodsonEffortBasedMC',      '20250508_Dodson_DodsonEffortBasedMC',  
                        '20250512_Dodson_DodsonEffortBasedMC',      '20250520_Dodson_DodsonEffortBasedMC',  
                        '20250523_Dodson_DodsonEffortBasedMC', 
                        
                        '20250507_Dodson_KangaEffortBasedMC',       '20250516_Dodson_KangaEffortBasedMC',   
                        '20250522_Dodson_KangaEffortBasedMC',   
                                   ]
        # self and other are corresponding to the recorded animals
        task_conditions = [
                            'self_EffortBasedSR', 'self_EffortBasedSR',
                            'self_EffortBasedSR', 'self_EffortBasedSR',
                            'self_EffortBasedSR', 'self_EffortBasedSR',
                            'self_EffortBasedSR', 'self_EffortBasedSR',
                            
                            'self_EffortBasedMC', 'self_EffortBasedMC',
                            'self_EffortBasedMC', 'self_EffortBasedMC',
                            'self_EffortBasedMC', 
            
            
                            'other_EffortBasedMC',  'other_EffortBasedMC',
                            'other_EffortBasedMC',  
                          ]
        dates_list = [
                        # both animals' lever force were changed - Self reward
                        "20250505",    "20250506_SR", "20250508_SR", "20250516_SR",
                        "20250519",    "20250520_SR", "20250522_SR", "20250523_SR",
                        # Dodson's lever force were changed
                        "20250506_MC", "20250508",    "20250512_MC", "20250520",
                        "20250523", 
                        # Kanga's lever force were changed
                        "20250507",    "20250516_MC", "20250522"
                     ]
        videodates_list = [
                            # "both animals' lever force were changed - Self reward
                            "20250505",    "20250506_SR", "20250508_SR", "20250516_SR",
                            "20250519",    "20250520_SR", "20250522_SR", "20250523_SR",
                            # Dodson's lever force were changed
                            "20250506_MC", "20250508",    "20250512_MC", "20250520",
                            "20250523", 
                            # Kanga's lever force were changed
                            "20250507",    "20250516_MC", "20250522"
                          ] # to deal with the sessions that MC and SR were in the same session
        session_start_times = [ 
                                147.1, 0.00, 0.00, 0.00,
                                106.0, 0.00, 0.00, 257.0,    
                                
                                0.00, 151.0, 265.0, 0.00, 
                                66.4,
            
                                0.00, 171.1, 0.00,                     
                              ] # in second
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0']*np.shape(dates_list)[0]
        
        animal1_fixedorders = ['dodson']*np.shape(dates_list)[0]
        animal2_fixedorders = ['kanga']*np.shape(dates_list)[0]

        animal1_filenames = ["Dodson"]*np.shape(dates_list)[0]
        animal2_filenames = ["Kanga"]*np.shape(dates_list)[0]
        
        recordedanimals = animal1_fixedorders 
        
    elif do_OFC:
        neural_record_conditions = [
                                
                                   ]
        task_conditions = [
                            
                          ]
        dates_list = [
                      
                     ]
        videodates_list = dates_list
        session_start_times = [ 
                                
                              ] # in second
        
        kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
        
        trig_channelnames = ['Dev1/ai0']*np.shape(dates_list)[0]
    
        animal1_fixedorders = ['dodson']*np.shape(dates_list)[0]
        animal2_fixedorders = ['kanga']*np.shape(dates_list)[0]

        animal1_filenames = ["Dodson"]*np.shape(dates_list)[0]
        animal2_filenames = ["Kanga"]*np.shape(dates_list)[0]
        
        recordedanimals = animal1_fixedorders 
    
# a test case
if 0:
    neural_record_conditions = ['20240910_Kanga_EffortBasedMC',]
    dates_list = ['20240910',]
    videodates_list = dates_list
    task_conditions = ['self_EffortBasedMC',]
    session_start_times = [0.00] # in second
    kilosortvers = list((np.ones(np.shape(dates_list))*4).astype(int))
    trig_channelnames = ['Dev1/ai0']*np.shape(dates_list)[0]
    animal1_fixedorders = ['dannon']
    animal2_fixedorders = ['kanga']
    animal1_filenames = ["Dannon"]
    animal2_filenames = ["Kanga"]
    recordedanimals = animal2_fixedorders 

    
    

ndates = np.shape(dates_list)[0]

session_start_frames = session_start_times * fps # fps is 30Hz

# video tracking results info
animalnames_videotrack = ['dodson','scorch'] # does not really mean dodson and scorch, instead, indicate animal1 and animal2
bodypartnames_videotrack = ['rightTuft','whiteBlaze','leftTuft','rightEye','leftEye','mouth']


# which camera to analyzed
cameraID = 'camera-2'
cameraID_short = 'cam2'


# location of levers and tubes for camera 2
# get this information using DLC animal tracking GUI, the results are stored: 
# /home/ws523/marmoset_tracking_DLCv2/marmoset_tracking_with_lever_tube-weikang-2023-04-13/labeled-data/
considerlevertube = 1
considertubeonly = 0
# # camera 1
# lever_locs_camI = {'dodson':np.array([645,600]),'scorch':np.array([425,435])}
# tube_locs_camI  = {'dodson':np.array([1350,630]),'scorch':np.array([555,345])}
# # camera 2
lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
tube_locs_camI  = {'dodson':np.array([1550,515]),'scorch':np.array([350,515])}
# # lever_locs_camI = {'dodson':np.array([1335,715]),'scorch':np.array([550,715])}
# # tube_locs_camI  = {'dodson':np.array([1650,490]),'scorch':np.array([250,490])}
# # camera 3
# lever_locs_camI = {'dodson':np.array([1580,440]),'scorch':np.array([1296,540])}
# tube_locs_camI  = {'dodson':np.array([1470,375]),'scorch':np.array([805,475])}


if np.shape(session_start_times)[0] != np.shape(dates_list)[0]:
    exit()


# GLM related variables
# all these are not in use...
Kernel_coefs_all_dates = dict.fromkeys(dates_list, [])
Kernel_spikehist_all_dates = dict.fromkeys(dates_list, [])
#
Kernel_coefs_all_shuffled_dates = dict.fromkeys(dates_list, [])
Kernel_spikehist_all_shuffled_dates = dict.fromkeys(dates_list, [])


# for summarizing spike related traces
spike_trig_trace_summary = pd.DataFrame(columns=[
                                            'dates', 'condition', 'act_animal', 'bhv_name',
                                        ])
bhv_aligned_FR_trace_summary = pd.DataFrame(columns=[
                                            'dates', 'condition', 'act_animal', 'bhv_name',
                                        ])

# for summarizing the encoding with LOO
neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary = []
neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary = dict.fromkeys(dates_list, [])


# where to save the summarizing data
data_saved_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/3d_recontruction_analysis_forceManipulation_task_data_saved/'

# neural data folder
neural_data_folder = '/gpfs/radev/pi/nandy/jadi_gibbs_data/Marmoset_neural_recording/'

    

In [ ]:
print(np.shape(neural_record_conditions))
print(np.shape(task_conditions))
print(np.shape(dates_list))
print(np.shape(videodates_list)) 
print(np.shape(session_start_times))

print(np.shape(kilosortvers))

print(np.shape(trig_channelnames))
print(np.shape(animal1_fixedorders)) 
print(np.shape(recordedanimals))
print(np.shape(animal2_fixedorders))

print(np.shape(animal1_filenames))
print(np.shape(animal2_filenames))  

In [ ]:
print(savefile_sufix)
print(recordedanimals[0])

In [ ]:
# basic behavior analysis (define time stamps for each bhv events, etc)

try:
    if redo_anystep:
        dummy
    
    # dummy
    
    # load saved data
    data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neuralGLM_new'+savefile_sufix+'/'+cameraID+'/'+ \
                           animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+recordedanimals[0]+'Recorded'+'/'
    

    # with open(data_saved_subfolder+'/spike_trig_trace_summary_'+glm_encoding_type+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
    #     spike_trig_trace_summary = pickle.load(f)         
    # with open(data_saved_subfolder+'/bhv_aligned_FR_trace_summary_'+glm_encoding_type+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
    #     bhv_aligned_FR_trace_summary = pickle.load(f) 
        
    with open(data_saved_subfolder+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary= pickle.load(f)         
    with open(data_saved_subfolder+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary = pickle.load(f) 
            
    
    
    print('all data from all dates are loaded')

except:

    print('analyze all dates')

    for idate in np.arange(0,ndates,1):
    
        date_tgt = dates_list[idate]
        videodate_tgt = videodates_list[idate]
        
        neural_record_condition = neural_record_conditions[idate]
        
        session_start_time = session_start_times[idate]
        
        kilosortver = kilosortvers[idate]

        trig_channelname = trig_channelnames[idate]
        
        animal1_filename = animal1_filenames[idate]
        animal2_filename = animal2_filenames[idate]
        
        animal1_fixedorder = [animal1_fixedorders[idate]]
        animal2_fixedorder = [animal2_fixedorders[idate]]
        
        recordedanimal = recordedanimals[idate]
        
        # load behavioral results
        try:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_forceManipulation_task/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path +date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_session_info_" + "*.json")
            ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal2_filename+"_"+animal1_filename+"_ni_data_" + "*.json")
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            # 
            with open(ni_data_json[0]) as f:
                for line in f:
                    ni_data=json.loads(line)   
        except:
            bhv_data_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/marmoset_tracking_bhv_data_forceManipulation_task/"+date_tgt+"_"+animal1_filename+"_"+animal2_filename+"/"
            trial_record_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_TrialRecord_" + "*.json")
            bhv_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_bhv_data_" + "*.json")
            session_info_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_session_info_" + "*.json")
            ni_data_json = glob.glob(bhv_data_path + date_tgt+"_"+animal1_filename+"_"+animal2_filename+"_ni_data_" + "*.json")
            #
            trial_record = pd.read_json(trial_record_json[0])
            bhv_data = pd.read_json(bhv_data_json[0])
            session_info = pd.read_json(session_info_json[0])
            #
            with open(ni_data_json[0]) as f:
                for line in f:
                    ni_data=json.loads(line)

        # get animal info from the session information
        animal1 = session_info['lever1_animal'][0].lower()
        animal2 = session_info['lever2_animal'][0].lower()

        
        # get task type and cooperation threshold
        try:
            coop_thres = session_info["pulltime_thres"][0]
            tasktype = session_info["task_type"][0]
        except:
            coop_thres = 0
            tasktype = 1
    
            
        # clean up the trial_record
        warnings.filterwarnings('ignore')
        trial_record_clean = pd.DataFrame(columns=trial_record.columns)
        # for itrial in np.arange(0,np.max(trial_record['trial_number']),1):
        for itrial in trial_record['trial_number']:
            # trial_record_clean.loc[itrial] = trial_record[trial_record['trial_number']==itrial+1].iloc[[0]]
            trial_record_clean = trial_record_clean.append(trial_record[trial_record['trial_number']==itrial].iloc[[0]])
        trial_record_clean = trial_record_clean.reset_index(drop = True)

        # change bhv_data time to the absolute time
        time_points_new = pd.DataFrame(np.zeros(np.shape(bhv_data)[0]),columns=["time_points_new"])
        # for itrial in np.arange(0,np.max(trial_record_clean['trial_number']),1):
        for itrial in np.arange(0,np.shape(trial_record_clean)[0],1):
            # ind = bhv_data["trial_number"]==itrial+1
            ind = bhv_data["trial_number"]==trial_record_clean['trial_number'][itrial]
            new_time_itrial = bhv_data[ind]["time_points"] + trial_record_clean["trial_starttime"].iloc[itrial]
            time_points_new["time_points_new"][ind] = new_time_itrial
        bhv_data["time_points"] = time_points_new["time_points_new"]
        bhv_data = bhv_data[bhv_data["time_points"] != 0]

        
        
        # load behavioral event results
        try:
            # dummy
            print('load social gaze with '+cameraID+' only of '+date_tgt)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'rb') as f:
                output_look_ornot = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'rb') as f:
                output_allvectors = pickle.load(f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'rb') as f:
                output_allangles = pickle.load(f)  
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'rb') as f:
                output_key_locations = pickle.load(f)
        except:   

            # folder and file path
            camera12_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera12/"
            camera23_analyzed_path = "/gpfs/radev/pi/nandy/jadi_gibbs_data/VideoTracker_SocialInter/test_video_cooperative_task_3d/"+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_camera23/"
            
            # 
            try: 
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
                bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                if not os.path.exists(bodyparts_camI_camIJ):
                    singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                    bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                if not os.path.exists(bodyparts_camI_camIJ):
                    singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                    bodyparts_camI_camIJ = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"                
                # get the bodypart data from files
                bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
                video_file_original = camera12_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"
            except:
                singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_167500"
                bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                if not os.path.exists(bodyparts_camI_camIJ):
                    singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_camera_withHeadchamberFeb28shuffle1_80000"
                    bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
                if not os.path.exists(bodyparts_camI_camIJ):
                    singlecam_ana_type = "DLC_dlcrnetms5_marmoset_tracking_with_middle_cameraSep1shuffle1_150000"
                    bodyparts_camI_camIJ = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+singlecam_ana_type+"_el_filtered.h5"
            
            # get the bodypart data from files
            bodyparts_locs_camI = body_part_locs_singlecam(bodyparts_camI_camIJ,singlecam_ana_type,animalnames_videotrack,bodypartnames_videotrack,videodate_tgt)
            video_file_original = camera23_analyzed_path+videodate_tgt+"_"+animal1_filename+"_"+animal2_filename+"_"+cameraID+".mp4"        
        
            
            print('analyze social gaze with '+cameraID+' only of '+date_tgt)
            # get social gaze information 
            output_look_ornot, output_allvectors, output_allangles = find_socialgaze_timepoint_singlecam_wholebody(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,
                                                                                                                   considerlevertube,considertubeonly,sqr_thres_tubelever,
                                                                                                                   sqr_thres_face,sqr_thres_body)
            output_key_locations = find_socialgaze_timepoint_singlecam_wholebody_2(bodyparts_locs_camI,lever_locs_camI,tube_locs_camI,considerlevertube)
            
            # save data
            current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody/'+animal1_fixedorder[0]+animal2_fixedorder[0]
            add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
            if not os.path.exists(add_date_dir):
                os.makedirs(add_date_dir)
            #
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_look_ornot.pkl', 'wb') as f:
                pickle.dump(output_look_ornot, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allvectors.pkl', 'wb') as f:
                pickle.dump(output_allvectors, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_allangles.pkl', 'wb') as f:
                pickle.dump(output_allangles, f)
            with open(data_saved_folder+"bhv_events_singlecam_wholebody/"+animal1_fixedorder[0]+animal2_fixedorder[0]+"/"+cameraID+'/'+date_tgt+'/output_key_locations.pkl', 'wb') as f:
                pickle.dump(output_key_locations, f)
                

        look_at_other_or_not_merge = output_look_ornot['look_at_other_or_not_merge']
        look_at_tube_or_not_merge = output_look_ornot['look_at_tube_or_not_merge']
        look_at_lever_or_not_merge = output_look_ornot['look_at_lever_or_not_merge']
        look_at_otherlever_or_not_merge = output_look_ornot['look_at_otherlever_or_not_merge']
        look_at_otherface_or_not_merge = output_look_ornot['look_at_otherface_or_not_merge']
        
        # change the unit to second and align to the start of the session
        session_start_time = session_start_times[idate]
        look_at_other_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_other_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_lever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_lever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_tube_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_tube_or_not_merge['dodson'])[0],1)/fps - session_start_time 
        look_at_otherlever_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherlever_or_not_merge['dodson'])[0],1)/fps - session_start_time
        look_at_otherface_or_not_merge['time_in_second'] = np.arange(0,np.shape(look_at_otherface_or_not_merge['dodson'])[0],1)/fps - session_start_time

        
        # find time point of behavioral events
        output_time_points_socialgaze ,output_time_points_levertube = bhv_events_timepoint_singlecam(bhv_data,look_at_other_or_not_merge,look_at_lever_or_not_merge,look_at_tube_or_not_merge)
        time_point_pull1 = output_time_points_socialgaze['time_point_pull1']
        time_point_pull2 = output_time_points_socialgaze['time_point_pull2']
        oneway_gaze1 = output_time_points_socialgaze['oneway_gaze1']
        oneway_gaze2 = output_time_points_socialgaze['oneway_gaze2']
        mutual_gaze1 = output_time_points_socialgaze['mutual_gaze1']
        mutual_gaze2 = output_time_points_socialgaze['mutual_gaze2']
        lever_gaze1 = output_time_points_levertube['time_point_lookatlever1']
        lever_gaze2 = output_time_points_levertube['time_point_lookatlever2']
        # 
        # mostly just for the sessions in which MC and SR are in the same session 
        firstpulltime = np.nanmin([np.nanmin(time_point_pull1),np.nanmin(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1>(firstpulltime-15)] # 15s before the first pull (animal1 or 2) count as the active period
        oneway_gaze2 = oneway_gaze2[oneway_gaze2>(firstpulltime-15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1>(firstpulltime-15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2>(firstpulltime-15)]  
        lever_gaze1 = lever_gaze1[lever_gaze1>(firstpulltime-15)]
        lever_gaze2 = lever_gaze2[lever_gaze2>(firstpulltime-15)]
        #    
        # newly added condition: only consider gaze during the active pulling time (15s after the last pull)    
        lastpulltime = np.nanmax([np.nanmax(time_point_pull1),np.nanmax(time_point_pull2)])
        oneway_gaze1 = oneway_gaze1[oneway_gaze1<(lastpulltime+15)]    
        oneway_gaze2 = oneway_gaze2[oneway_gaze2<(lastpulltime+15)]
        mutual_gaze1 = mutual_gaze1[mutual_gaze1<(lastpulltime+15)]
        mutual_gaze2 = mutual_gaze2[mutual_gaze2<(lastpulltime+15)] 
        lever_gaze1 = lever_gaze1[lever_gaze1<(lastpulltime+15)] 
        lever_gaze2 = lever_gaze2[lever_gaze2<(lastpulltime+15)] 
            
        # define successful pulls and failed pulls
        # a new definition of successful and failed pulls
        # separate successful and failed pulls
        # step 1 all pull and juice
        time_point_pull1 = bhv_data["time_points"][bhv_data["behavior_events"]==1]
        time_point_pull2 = bhv_data["time_points"][bhv_data["behavior_events"]==2]
        time_point_juice1 = bhv_data["time_points"][bhv_data["behavior_events"]==3]
        time_point_juice2 = bhv_data["time_points"][bhv_data["behavior_events"]==4]
        # step 2:
        # pull 1
        # Find the last pull before each juice
        successful_pull1 = [time_point_pull1[time_point_pull1 < juice].max() for juice in time_point_juice1]
        # Convert to Pandas Series
        successful_pull1 = pd.Series(successful_pull1, index=time_point_juice1.index)
        # Find failed pulls (pulls that are not successful)
        failed_pull1 = time_point_pull1[~time_point_pull1.isin(successful_pull1)]
        # pull 2
        # Find the last pull before each juice
        successful_pull2 = [time_point_pull2[time_point_pull2 < juice].max() for juice in time_point_juice2]
        # Convert to Pandas Series
        successful_pull2 = pd.Series(successful_pull2, index=time_point_juice2.index)
        # Find failed pulls (pulls that are not successful)
        failed_pull2 = time_point_pull2[~time_point_pull2.isin(successful_pull2)]
        #
        # step 3:
        time_point_pull1_succ = np.round(successful_pull1,1)
        time_point_pull2_succ = np.round(successful_pull2,1)
        time_point_pull1_fail = np.round(failed_pull1,1)
        time_point_pull2_fail = np.round(failed_pull2,1)
        # 
        time_point_pulls_succfail = { "pull1_succ":time_point_pull1_succ,
                                      "pull2_succ":time_point_pull2_succ,
                                      "pull1_fail":time_point_pull1_fail,
                                      "pull2_fail":time_point_pull2_fail,
                                    }
        
        # new total session time (instead of 600s) - total time of the video recording
        totalsess_time = np.floor(np.shape(output_look_ornot['look_at_lever_or_not_merge']['dodson'])[0]/30) 
        #
        # remove task irrelavant period
        if totalsess_time > (lastpulltime+session_start_time+15):
            totalsess_time = np.floor(lastpulltime+session_start_time+15)
            
        
        # newly added info for lever1 and lever2's self force (delta force compared to the previous block)
        def get_force_changes(df, force_col):
            force_series = df[force_col]
            change_mask = force_series != force_series.shift(1)
            change_mask.iloc[0] = False  # exclude first row
            changes = df[change_mask]
            #
            # First block always has delta = 0
            first_time = df['trial_starttime'].iloc[0]
            result = [[0, first_time]]
            #
            if changes.empty:
                return np.array(result)
            #
            for idx in changes.index:
                current_force = df.loc[idx, force_col]
                prev_force = force_series.iloc[force_series.index.get_loc(idx) - 1]
                delta = current_force - prev_force
                time = df.loc[idx, 'trial_starttime']
                result.append([delta, time])
            #
            return np.array(result)

        lever1_delta_force = get_force_changes(trial_record_clean, 'lever1_force')
        lever2_delta_force = get_force_changes(trial_record_clean, 'lever2_force')   
        
        
        
        # session starting time compared with the neural recording
        session_start_time_niboard_offset = ni_data['session_t0_offset'] # in the unit of second
        try:
            neural_start_time_niboard_offset = ni_data['trigger_ts'][0]['elapsed_time'] # in the unit of second
        except: # for the multi-animal recording setup
            neural_start_time_niboard_offset = next(
                entry['timepoints'][0]['elapsed_time']
                for entry in ni_data['trigger_ts']
                if entry['channel_name'] == f"{trig_channelname}")
        neural_start_time_session_start_offset = neural_start_time_niboard_offset-session_start_time_niboard_offset
    
    
    
        # load channel maps
        channel_map_file = '/home/ws523/kilisort_spikesorting/Channel-Maps/Neuronexus_whitematter_2x32.mat'
        # channel_map_file = '/home/ws523/kilisort_spikesorting/Channel-Maps/Neuronexus_whitematter_2x32_kilosort4_new.mat'
        channel_map_data = scipy.io.loadmat(channel_map_file)
            
        # # load spike sorting results
        if 1:
            print('load spike data for '+neural_record_condition)
            if kilosortver == 2:
                spike_time_file = neural_data_folder+neural_record_condition+'/Kilosort/spike_times.npy'
                spike_time_data = np.load(spike_time_file)
            elif kilosortver == 4:
                spike_time_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/spike_times.npy'
                spike_time_data = np.load(spike_time_file)
            # 
            # align the FR recording time stamps
            spike_time_data = spike_time_data + fs_spikes*neural_start_time_session_start_offset
            # down-sample the spike recording resolution to 30Hz
            spike_time_data = spike_time_data/fs_spikes*fps
            spike_time_data = np.round(spike_time_data)
            #
            if kilosortver == 2:
                spike_clusters_file = neural_data_folder+neural_record_condition+'/Kilosort/spike_clusters.npy'
                spike_clusters_data = np.load(spike_clusters_file)
                spike_channels_data = np.copy(spike_clusters_data)
            elif kilosortver == 4:
                spike_clusters_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/spike_clusters.npy'
                spike_clusters_data = np.load(spike_clusters_file)
                spike_channels_data = np.copy(spike_clusters_data)
            #
            if kilosortver == 2:
                channel_maps_file = neural_data_folder+neural_record_condition+'/Kilosort/channel_map.npy'
                channel_maps_data = np.load(channel_maps_file)
            elif kilosortver == 4:
                channel_maps_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/channel_map.npy'
                channel_maps_data = np.load(channel_maps_file)
            #
            if kilosortver == 2:
                channel_pos_file = neural_data_folder+neural_record_condition+'/Kilosort/channel_positions.npy'
                channel_pos_data = np.load(channel_pos_file)
            elif kilosortver == 4:
                channel_pos_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/channel_positions.npy'
                channel_pos_data = np.load(channel_pos_file)
            #
            if kilosortver == 2:
                clusters_info_file = neural_data_folder+neural_record_condition+'/Kilosort/cluster_info.tsv'
                clusters_info_data = pd.read_csv(clusters_info_file,sep="\t")
            elif kilosortver == 4:
                clusters_info_file = neural_data_folder+neural_record_condition+'/kilosort4_6500HzNotch/cluster_info.tsv'
                clusters_info_data = pd.read_csv(clusters_info_file,sep="\t")
            #
            # only get the spikes that are manually checked
            try:
                good_clusters = clusters_info_data[(clusters_info_data.group=='good')|(clusters_info_data.group=='mua')]['cluster_id'].values
            except:
                good_clusters = clusters_info_data[(clusters_info_data.group=='good')|(clusters_info_data.group=='mua')]['id'].values
            #
            clusters_info_data = clusters_info_data[~pd.isnull(clusters_info_data.group)]
            #
            spike_time_data = spike_time_data[np.isin(spike_clusters_data,good_clusters)]
            spike_channels_data = spike_channels_data[np.isin(spike_clusters_data,good_clusters)]
            spike_clusters_data = spike_clusters_data[np.isin(spike_clusters_data,good_clusters)]
            
            #
            nclusters = np.shape(clusters_info_data)[0]
            #
            for icluster in np.arange(0,nclusters,1):
                try:
                    cluster_id = clusters_info_data['id'].iloc[icluster]
                except:
                    cluster_id = clusters_info_data['cluster_id'].iloc[icluster]
                spike_channels_data[np.isin(spike_clusters_data,cluster_id)] = clusters_info_data['ch'].iloc[icluster]   
            # 
            # get the channel to depth information, change 2 shanks to 1 shank 
            try:
                channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1]*2,channel_pos_data[channel_pos_data[:,0]==1,1]*2+1])
                # channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1],channel_pos_data[channel_pos_data[:,0]==31.2,1]])            
                # channel_to_depth = np.vstack([channel_maps_data.T[0],channel_depth])
                channel_to_depth = np.vstack([channel_maps_data.T,channel_depth])
            except:
                channel_depth=np.hstack([channel_pos_data[channel_pos_data[:,0]==0,1],channel_pos_data[channel_pos_data[:,0]==31.2,1]])            
                # channel_to_depth = np.vstack([channel_maps_data.T[0],channel_depth])
                channel_to_depth = np.vstack([channel_maps_data.T,channel_depth])
                channel_to_depth[1] = channel_to_depth[1]/30-64 # make the y axis consistent
        
        
        #
        # get the dataset for GLM and run GLM
        if 1:
            # get the organized data for GLM
            print('get '+neural_record_condition+' data for single camera GLM fitting')
            #
            gausKernelsize = 4 # 4; 15
            
            data_summary_twoanimals, data_summary_names, spiketrain_summary = get_singlecam_bhv_var_for_neuralGLM_fitting_BasisKernelsForContVaris(gausKernelsize,fps, 
                                                                                        animal1, animal2, recordedanimal, animalnames_videotrack, 
                                                                                        session_start_time, time_point_pull1, time_point_pull2, time_point_juice1, time_point_juice2, 
                                                                                        oneway_gaze1, oneway_gaze2, mutual_gaze1, mutual_gaze2,
                                                                                        lever1_delta_force, lever2_delta_force,
                                                                                        output_look_ornot, output_allvectors, output_allangles, output_key_locations, 
                                                                                        spike_clusters_data, spike_time_data, spike_channels_data)
                
            data_summary = data_summary_twoanimals[recordedanimal]
            
            
            # MODIFICATION: Define kernel parameters here for easy adjustment
            KERNEL_DURATION_S = 8.0  # total span: -4s to +4s
            KERNEL_OFFSET_S = -4.0   # shift so it starts at -4s
            N_BASIS_FUNCS = 26     # The number of basis functions to represent the kernel
                
            var_toglm_names = ['gaze_other_angle', 'gaze_lever_angle', # 'gaze_tube_angle',
                               'animal_animal_dist', 'animal_lever_dist', # 'animal_tube_dist',
                               'mass_move_speed', 'gaze_angle_speed',
                               'otherani_otherlever_dist', # 'otherani_othertube_dist', # 'othergaze_self_angle',
                               'other_mass_move_speed',
                               'selfpull_prob',
                               'socialgaze_prob',
                               'otherpull_prob',
                               'self_deltaforce',
                               'other_deltaforce'
                              ]
            nvars_toglm = np.shape(var_toglm_names)[0]
                        
                        
            #
            # neuralGLM with the three axis themselves + the partner's PC1 + self and partner's delta force 
            try:
                var_toglm_names_mainAxes_partnerPC1 = ['var_pull', 'var_gaze', 'var_juice', 'partner_PC1',]
                
                # dummy
                
                print('load the session wised data for neural GLM fitting - pull gaze juice axes themselves + partner pc1 neural-glm' )

                current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_neuralglm_model'+savefile_sufix+'/'+\
                              animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+recordedanimal+'Recorded'
                add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)

                with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_coef.pkl', 'rb') as f:
                    neuralGLM_mainAxes_partnerPC1_kernels_coef = pickle.load(f)
                with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_tempFilter.pkl', 'rb') as f:
                    neuralGLM_mainAxes_partnerPC1_kernels_tempFilter = pickle.load(f)
                with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_coef_shf.pkl', 'rb') as f:
                    neuralGLM_mainAxes_partnerPC1_kernels_coef_shf = pickle.load(f)
                with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_tempFilter_shf.pkl', 'rb') as f:
                    neuralGLM_mainAxes_partnerPC1_kernels_tempFilter_shf = pickle.load(f)
                #
                with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_scalar_coef.pkl', 'rb') as f:
                    neuralGLM_mainAxes_partnerPC1_scalar_coef = pickle.load(f)
                with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_scalar_coef_shf.pkl', 'rb') as f:
                    neuralGLM_mainAxes_partnerPC1_scalar_coef_shf = pickle.load(f)
                
            except:
                
                print('do GLM fitting for spike trains with continuous variables - pull gaze juice axes themselves + partner pc1 neural-glm')
                
                # dp the glm for n bootstraps, each bootstrap do 80/20 training/testing
                N_BOOTSTRAPS = 100
                test_size = 0.4
                #
                dospikehist = 0
                spikehist_twin = 2
                #
                try:
                    neuralGLM_mainAxes_partnerPC1_kernels_coef, neuralGLM_mainAxes_partnerPC1_kernels_tempFilter, \
                    neuralGLM_mainAxes_partnerPC1_kernels_coef_shf, neuralGLM_mainAxes_partnerPC1_kernels_tempFilter_shf, \
                    neuralGLM_mainAxes_partnerPC1_scalar_coef, neuralGLM_mainAxes_partnerPC1_scalar_coef_shf, \
                    var_toglm_names_mainAxes_partnerPC1 = neuralGLM_fitting_BasisKernelsForContVaris_PullGazeAxis_partnerPC1(
                                                    KERNEL_DURATION_S, KERNEL_OFFSET_S,
                                                    N_BASIS_FUNCS, fps, 
                                                    animal1, animal2, recordedanimal,
                                                    var_toglm_names, data_summary_names, data_summary_twoanimals, 
                                                    spiketrain_summary, dospikehist, spikehist_twin, 
                                                    N_BOOTSTRAPS,test_size )

                    # save data
                    if 1:
                        current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_neuralglm_model'+savefile_sufix+'/'+\
                                      animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+recordedanimal+'Recorded'
                        add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
                        if not os.path.exists(add_date_dir):
                            os.makedirs(add_date_dir)
                        #
                        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_coef.pkl', 'wb') as f:
                            pickle.dump(neuralGLM_mainAxes_partnerPC1_kernels_coef, f)
                        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_tempFilter.pkl', 'wb') as f:
                            pickle.dump(neuralGLM_mainAxes_partnerPC1_kernels_tempFilter, f)
                        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_coef_shf.pkl', 'wb') as f:
                            pickle.dump(neuralGLM_mainAxes_partnerPC1_kernels_coef_shf, f)
                        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_tempFilter_shf.pkl', 'wb') as f:
                            pickle.dump(neuralGLM_mainAxes_partnerPC1_kernels_tempFilter_shf, f)
                        #
                        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_scalar_coef.pkl', 'wb') as f:
                            pickle.dump(neuralGLM_mainAxes_partnerPC1_scalar_coef, f)
                        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_scalar_coef_shf.pkl', 'wb') as f:
                            pickle.dump(neuralGLM_mainAxes_partnerPC1_scalar_coef_shf, f)
                        
                except:
                    # continue
                    print('error! skip')
                        
            
            #
            # neuralGLM with the three axis themselves + the partner's PC1; use leave-one-out method
            try:
                var_toglm_names_split = ['var_pull_past', 'var_pull_future', 'var_gaze_past', 'var_gaze_future', 
                                         'var_juice_past', 'var_juice_future', 'partner_PC1_past', 'partner_PC1_future']
                
                # dummy
                
                print('load the session wised data for neural GLM fitting - pull gaze juice axes themselves + partner pc1 neural-glm; leave-one-out method' )

                current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_neuralglm_model'+savefile_sufix+'/'+\
                              animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+recordedanimal+'Recorded'
                add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)

                with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues.pkl', 'rb') as f:
                    neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues = pickle.load(f)
                with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter.pkl', 'rb') as f:
                    neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter = pickle.load(f)
                
            except:
                
                print('do GLM fitting for spike trains with continuous variables - pull gaze juice axes themselves + partner pc1 neural-glm; leave-one-out method')
                
                # dp the glm for n bootstraps, each bootstrap do 80/20 training/testing
                N_BOOTSTRAPS = 100
                test_size = 0.4
                #
                dospikehist = 0
                spikehist_twin = 2
                #
                # try:
                neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues, \
                neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter, \
                var_toglm_names_split = \
                neuralGLM_fitting_BasisKernelsForContVaris_PullGazeAxis_partnerPC1_LOOmethods(
                                                KERNEL_DURATION_S, KERNEL_OFFSET_S,
                                                N_BASIS_FUNCS, fps, 
                                                animal1, animal2, recordedanimal,
                                                var_toglm_names, data_summary_names, data_summary_twoanimals, 
                                                spiketrain_summary, dospikehist, spikehist_twin, 
                                                N_BOOTSTRAPS,test_size )
                    
                # save data
                if 1:
                    current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_neuralglm_model'+savefile_sufix+'/'+\
                                  animal1_fixedorder[0]+animal2_fixedorder[0]+'_'+recordedanimal+'Recorded'
                    add_date_dir = os.path.join(current_dir,cameraID+'/'+date_tgt)
                    if not os.path.exists(add_date_dir):
                        os.makedirs(add_date_dir)
                    #
                    with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues.pkl', 'wb') as f:
                        pickle.dump(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues, f)
                    with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter.pkl', 'wb') as f:
                        pickle.dump(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter, f)
                
                # except:
                #     # continue
                #     print('error! skip')
                        
            #     
            # add more information to the pvalue and tempfilter
            #
            df = neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues
            df.index.name = 'neuronID'
            df.reset_index(inplace=True) # inplace=True modifies the DataFrame directly
            # Add a new column called 'date' with today's date
            df['date'] = date_tgt
                                     
            # put in the summary 
            neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary.append(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues)
            # 
            neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary[date_tgt] = neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter
                        
                        
                    
            

                
            

    # save data
    if 0:
        
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neuralGLM_new'+savefile_sufix+'/'+cameraID+'/'+ \
                           animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+recordedanimals[0]+'Recorded'+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)       
          
        # with open(data_saved_subfolder+'/spike_trig_trace_summary_'+glm_encoding_type+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
        #     pickle.dump(spike_trig_trace_summary, f)    
        with open(data_saved_subfolder+'/bhv_aligned_FR_trace_summary_'+glm_encoding_type+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(bhv_aligned_FR_trace_summary, f)   
        
        with open(data_saved_subfolder+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary, f)  
        
        # neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary = pd.concat(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary, 
        #                                                                                 ignore_index=True)
        
        with open(data_saved_subfolder+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary, f)  


    # save a subset of data
    if 0:
        
        data_saved_subfolder = data_saved_folder+'data_saved_singlecam_wholebody_neuralGLM_new'+savefile_sufix+'/'+cameraID+'/'+ \
                           animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+recordedanimals[0]+'Recorded'+'/'
        if not os.path.exists(data_saved_subfolder):
            os.makedirs(data_saved_subfolder)       
        
        with open(data_saved_subfolder+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary, f)  
        
        # neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary = pd.concat(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary, 
        #                                                                                 ignore_index=True)
        with open(data_saved_subfolder+'/neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'wb') as f:
            pickle.dump(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary, f)  
    
    
    

In [ ]:
# Combine the list of DataFrames into a single DataFrame
try: 
    combined_df = pd.concat(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary, ignore_index=True)
    neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary = combined_df
except:
    print('neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary seems right')

In [ ]:
# example plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# --- 1. Define Parameters ---

# Total number of time points for the simulation
time_points = 1200
# The standard deviation (sigma) for the Gaussian kernel.
# This value controls the amount of smoothing.
# Try changing it to see how it affects the output!
sigma = 15.0

# --- 2. Generate Example Binary Data ---

# Create a time axis from 0 to time_points-1
time = np.arange(time_points)

# Create three empty binary arrays (all zeros)
pull_series = np.zeros(time_points)
gaze_series = np.zeros(time_points)
juice_series = np.zeros(time_points)

# Define the time points where events (spikes) occur for each series
pull_event_times = [200, 600, 1000]
gaze_event_times = [300, 700, 750, 950, 965, 980, 990]
juice_event_times = [400, 800, 1100]

# Set the values at the event times to 1.0 in our binary arrays
pull_series[pull_event_times] = 1.0
gaze_series[gaze_event_times] = 1.0
juice_series[juice_event_times] = 1.0

# --- 3. Apply Gaussian Smoothing ---

# Use scipy's gaussian_filter1d to efficiently smooth the data.
# This function applies a Gaussian filter along the array.
pull_smoothed = gaussian_filter1d(pull_series, sigma=sigma)
gaze_smoothed = gaussian_filter1d(gaze_series, sigma=sigma)
juice_smoothed = gaussian_filter1d(juice_series, sigma=sigma)

# --- 4. Plot the Results ---

# Create a figure with 3 subplots, arranged vertically.
# `sharex=True` ensures all plots share the same time axis.
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
fig.suptitle('Binary Time Series and Gaussian Smoothing', fontsize=16)

# Plotting settings for each series
plot_configs = [
    {
        'ax': axes[0],
        'title': 'Pull Axis Projection',
        'binary_series': pull_series,
        'smoothed_series': pull_smoothed,
        'binary_color': 'black',
        'linestyle': 'solid'
    },
    {
        'ax': axes[1],
        'title': 'Gaze Axis Projection',
        'binary_series': gaze_series,
        'smoothed_series': gaze_smoothed,
        'binary_color': '#8B4513', # SaddleBrown
        'linestyle': 'solid'
    },
    {
        'ax': axes[2],
        'title': 'Juice Axis Projection',
        'binary_series': juice_series,
        'smoothed_series': juice_smoothed,
        'binary_color': 'black',
        'linestyle': 'dashed'
    }
]

# Loop through the configurations to create each plot
for config in plot_configs:
    ax = config['ax']
    binary_series = config['binary_series']
    
    # Find the indices where events occur to plot the vertical lines
    event_indices = np.where(binary_series > 0)[0]
    
    # Plot the original binary events as vertical lines
    ax.vlines(event_indices, 0, 1,
              color=config['binary_color'],
              linestyle=config['linestyle'],
              linewidth=2.5,
              label='Binary Events')
              
    # Plot the smoothed curve
    ax.plot(time, config['smoothed_series'],
            color='dodgerblue',
            linewidth=2,
            label=f'Smoothed (σ={sigma})')

    # --- Formatting for a clean look ---
    ax.set_ylabel(config['title'], fontsize=12, rotation=0, labelpad=80)
    ax.set_yticks([]) # Remove y-axis ticks
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_position('center')
    ax.set_ylim(0, np.max(gaze_smoothed) * 1.1) # Ensure all curves are visible

# Set a label for the shared x-axis
axes[-1].set_xlabel('Time', fontsize=12)

# Adjust layout to prevent labels from overlapping
plt.tight_layout(rect=[0, 0, 1, 0.96]) # Adjust for suptitle
# plt.show()

savefig = 0
if savefig:
    figsavefolder = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents_neuralGLMfitting_BasisKernelsForContVaris_singlecam/"+\
                    cameraID+"/"

    if not os.path.exists(figsavefolder):
        os.makedirs(figsavefolder)

    fig.savefig(figsavefolder+'PullGazeJuice_binary_and_smoothed_example.pdf')




### do summarizing plot

In [ ]:
# do a summarizing plot based on spike_trig_trace_summary and bhv_trig_trace_summary
###

# act_animal_to_ana = 'kanga'
act_animal_to_ana = 'dodson'


conditions_to_ana = ['self_EffortBasedMC' ]
# self_EffortBasedSR, self_EffortBasedMC, other_EffortBasedMC


cond_toplot_type = 'self_EffortBasedMC'

In [ ]:
recordedanimals[0]

## plot using the loo methods to define the significant neurons 

In [ ]:
from statsmodels.stats.multitest import multipletests

# from p value, defining significant encoding type considering correction
indices = np.where(np.isin(task_conditions, conditions_to_ana))[0]
dates_to_ana = np.array(dates_list)[indices]

# load targeted pvalues
ind_con_tgt = np.isin(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary['date'],dates_to_ana) 
pvalue_tgt = neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_pvalues_summary[ind_con_tgt].copy()

# load targeted filters
tempFilter_tgt = {
    date: neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary[date]
    for date in dates_to_ana
    if date in neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary}


# Create a unique identifier for each neuron to handle data from different date
pvalue_tgt['unique_neuron'] = pvalue_tgt['date'].astype(str) + '_' + pvalue_tgt['neuronID'].astype(str)

# --- Step 1: Isolate the p-value columns for correction ---
p_value_cols = [
    'var_pull_past', 'var_pull_future', 'var_gaze_past', 'var_gaze_future',
    'var_juice_past', 'var_juice_future', 'partner_PC1_past', 'partner_PC1_future',
    'self_deltaforce', 'other_deltaforce',
]
raw_p_values_df = pvalue_tgt[p_value_cols]


# --- Step 2: Perform the Global FDR Correction ---

# Flatten the DataFrame into a single 1D array.
original_shape = raw_p_values_df.shape
all_p_values = raw_p_values_df.values.flatten()

# The correction function cannot handle NaNs, so we create a mask to track them.
nan_mask = np.isnan(all_p_values)
p_values_for_correction = all_p_values[~nan_mask]

# Apply the Benjamini-Hochberg FDR correction.
is_significant_flat, q_values_flat, _, _ = multipletests(
    p_values_for_correction, 
    alpha=0.05,  # Standard significance level for FDR
    method='fdr_bh' ## note: this is the one i used for the paper
    # method='bonferroni'
)


# Reshape the corrected q-values back into the original DataFrame shape.
q_values_full = np.full(all_p_values.shape, np.nan)
q_values_full[~nan_mask] = q_values_flat
q_values_reshaped = q_values_full.reshape(original_shape)

# Create a final DataFrame for the q-values with the original index and columns.
q_values_df = pd.DataFrame(q_values_reshaped, index=raw_p_values_df.index, columns=raw_p_values_df.columns)

# This is the fully corrected boolean DataFrame indicating significance (q < 0.05).
significant_results_df = q_values_df < 0.05
# significant_results_df = q_values_df < 0.1


# --- Step 3: Classify Encoding Type for ALL Variables ---

# Define a GENERALIZED function to classify based on corrected significance
def classify_encoding(row, base_variable_name):
    """
    Classifies encoding type for a given base variable name.
    Example: base_variable_name = 'var_pull'
    """
    sig_past = row[f'{base_variable_name}_past']
    sig_future = row[f'{base_variable_name}_future']
    
    if sig_past and sig_future:
        return 'Both'
    elif sig_past:
        return 'Reactive'
    elif sig_future:
        return 'Predictive'
    else:
        return 'None'
    
    
# Add a simple classifier for scalar variables    
def classify_encoding_scalar(row, variable_name):
    """
    Classifies encoding for scalar (non-kernel) variables like delta force.
    """
    return 'Significant' if row[variable_name] else 'None'


# Apply the function for each of the four behavioral variables
pull_encoding_type = significant_results_df.apply(lambda row: classify_encoding(row, 'var_pull'), axis=1)
gaze_encoding_type = significant_results_df.apply(lambda row: classify_encoding(row, 'var_gaze'), axis=1)
juice_encoding_type = significant_results_df.apply(lambda row: classify_encoding(row, 'var_juice'), axis=1)
partner_encoding_type = significant_results_df.apply(lambda row: classify_encoding(row, 'partner_PC1'), axis=1)

self_deltaforce_encoding = significant_results_df.apply(lambda row: classify_encoding_scalar(row, 'self_deltaforce'), axis=1)
other_deltaforce_encoding = significant_results_df.apply(lambda row: classify_encoding_scalar(row, 'other_deltaforce'), axis=1)


# --- Step 4: Create the Final, Expanded Summary DataFrame ---

summary_df = pd.DataFrame({
    'unique_neuron': pvalue_tgt['unique_neuron'],
    'pull_encoding': pull_encoding_type,
    'gaze_encoding': gaze_encoding_type,
    'juice_encoding': juice_encoding_type,
    'partner_PC1_encoding': partner_encoding_type,
    'self_deltaforce_encoding': self_deltaforce_encoding,
    'other_deltaforce_encoding': other_deltaforce_encoding
})


In [ ]:
np.shape(summary_df)

In [ ]:
# venn diagram
if 1:

    from matplotlib_venn import venn2, venn3

    # --- force to look ---
    # force_to_look = 'self_deltaforce'  # change to 'other_deltaforce' for partner force
    # force_to_look = 'other_deltaforce'
    if cond_toplot_type == 'other_EffortBasedMC':
        force_to_look = 'other_deltaforce'
    else:
        force_to_look = 'self_deltaforce'

    # --- Create a single figure to hold all plots ---
    fig = plt.figure(figsize=(15, 15))
    fig.suptitle('Analysis of Neuron Encoding Overlap', fontsize=22, fontweight='bold')

    # --- 1. Venn Diagrams for Reactive vs. Predictive Encoding ---
    def plot_encoding_type_venn(ax, df, base_name, title):
        reactive_set = set(df[df[f'{base_name}_encoding'].isin(['Reactive', 'Both'])]['unique_neuron'])
        predictive_set = set(df[df[f'{base_name}_encoding'].isin(['Predictive', 'Both'])]['unique_neuron'])
        none_count = df[df[f'{base_name}_encoding'] == 'None'].shape[0]
        venn2([reactive_set, predictive_set], set_labels=('Reactive', 'Predictive'), ax=ax)
        ax.set_title(title, fontsize=16)
        ax.text(0.95, 0.05, f"None: {none_count}",
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=12, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))

    ax1 = plt.subplot2grid((3, 3), (0, 0))
    ax2 = plt.subplot2grid((3, 3), (0, 1))
    ax3 = plt.subplot2grid((3, 3), (0, 2))
    plot_encoding_type_venn(ax1, summary_df, 'pull', 'Pull Encoding')
    plot_encoding_type_venn(ax2, summary_df, 'gaze', 'Gaze Encoding')
    plot_encoding_type_venn(ax3, summary_df, 'partner_PC1', 'Partner PC1 Encoding')

    # --- 2. Venn Diagrams for Overlap Between Variables ---
    def get_any_encoding_set(df, base_name):
        return set(df[df[f'{base_name}_encoding'] != 'None']['unique_neuron'])

    pull_set = get_any_encoding_set(summary_df, 'pull')
    gaze_set = get_any_encoding_set(summary_df, 'gaze')
    partner_set = get_any_encoding_set(summary_df, 'partner_PC1')

    ax4 = plt.subplot2grid((3, 3), (1, 0))
    ax5 = plt.subplot2grid((3, 3), (1, 1))
    ax6 = plt.subplot2grid((3, 3), (1, 2))

    none_pull_gaze = summary_df[(summary_df['pull_encoding'] == 'None') & (summary_df['gaze_encoding'] == 'None')].shape[0]
    venn2([pull_set, gaze_set], set_labels=('Pull', 'Gaze'), ax=ax4)
    ax4.set_title('Overlap: Pull vs. Gaze', fontsize=16)
    ax4.text(0.95, 0.05, f"None in both: {none_pull_gaze}",
             transform=ax4.transAxes, ha='right', va='bottom',
             fontsize=12, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))

    none_partner_gaze = summary_df[(summary_df['partner_PC1_encoding'] == 'None') & (summary_df['gaze_encoding'] == 'None')].shape[0]
    venn2([partner_set, gaze_set], set_labels=('Partner PC1', 'Gaze'), ax=ax5)
    ax5.set_title('Overlap: Partner PC1 vs. Gaze', fontsize=16)
    ax5.text(0.95, 0.05, f"None in both: {none_partner_gaze}",
             transform=ax5.transAxes, ha='right', va='bottom',
             fontsize=12, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))

    none_partner_pull = summary_df[(summary_df['partner_PC1_encoding'] == 'None') & (summary_df['pull_encoding'] == 'None')].shape[0]
    venn2([partner_set, pull_set], set_labels=('Partner PC1', 'Pull'), ax=ax6)
    ax6.set_title('Overlap: Partner PC1 vs. Pull', fontsize=16)
    ax6.text(0.95, 0.05, f"None in both: {none_partner_pull}",
             transform=ax6.transAxes, ha='right', va='bottom',
             fontsize=12, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))

    # --- 3. 3-Way Venn Diagram ---
    # --- define significant_all_three_df early since it's needed for the venn ---
    significant_all_three_df = summary_df[
        (summary_df['pull_encoding'] != 'None') &
        (summary_df['gaze_encoding'] != 'None') &
        (summary_df['partner_PC1_encoding'] != 'None')
    ].copy()
    
    ax7 = plt.subplot2grid((3, 3), (2, 0), colspan=2)
    none_all_three = summary_df[
        (summary_df['pull_encoding'] == 'None') &
        (summary_df['gaze_encoding'] == 'None') &
        (summary_df['partner_PC1_encoding'] == 'None')
    ].shape[0]
    venn3([pull_set, gaze_set, partner_set], set_labels=('Pull Encoding', 'Gaze Encoding', 'Partner PC1 Encoding'), ax=ax7)
    ax7.set_title('3-Way Overlap Between All Variables', fontsize=16)
    ax7.text(0.95, 0.05, f"None in all three: {none_all_three}",
             transform=ax7.transAxes, ha='right', va='bottom',
             fontsize=12, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))

    # new venn: triple-encoded vs force encoding
    # significant_all_three_df already defined above
    
    ax8_new = plt.subplot2grid((3, 3), (2, 2))
    triple_set = set(significant_all_three_df['unique_neuron'])
    force_set = get_any_encoding_set(summary_df, force_to_look)
    none_triple_force = summary_df[
        ~summary_df['unique_neuron'].isin(triple_set) &
        ~summary_df['unique_neuron'].isin(force_set)
    ].shape[0]
    venn2([triple_set, force_set],
          set_labels=('Triple Encoding', force_to_look),
          ax=ax8_new)
    ax8_new.set_title(f'Triple Encoding vs {force_to_look}', fontsize=16)
    ax8_new.text(0.95, 0.05, f"None in both: {none_triple_force}",
                 transform=ax8_new.transAxes, ha='right', va='bottom',
                 fontsize=12, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))

    
    # --- 4. Triple-encoded neurons figure ---
    

    # --- How many triple-encoded neurons also encode force? ---
    triple_and_force_df = significant_all_three_df[
        significant_all_three_df[f'{force_to_look}_encoding'] != 'None'
    ].copy()

    n_triple = len(significant_all_three_df)
    n_triple_and_force = len(triple_and_force_df)
    percent_triple_and_force = (n_triple_and_force / n_triple * 100) if n_triple > 0 else 0

    print(f"Triple-encoded neurons: {n_triple}")
    print(f"Triple-encoded + {force_to_look}: {n_triple_and_force} ({percent_triple_and_force:.1f}%)")

    fig2 = plt.figure(figsize=(15, 10))
    fig2.suptitle('Encoding Overlap for Neurons Significant in Pull, Gaze, AND Partner PC1', fontsize=22, fontweight='bold')

    ax8 = plt.subplot(2, 3, 1)
    ax9 = plt.subplot(2, 3, 2)
    ax10 = plt.subplot(2, 3, 3)
    plot_encoding_type_venn(ax8, significant_all_three_df, 'pull', 'Pull Encoding (in triple-coded neurons)')
    plot_encoding_type_venn(ax9, significant_all_three_df, 'gaze', 'Gaze Encoding (in triple-coded neurons)')
    plot_encoding_type_venn(ax10, significant_all_three_df, 'partner_PC1', 'Partner PC1 Encoding (in triple-coded neurons)')

    significant_pull_df = summary_df[
        (summary_df['pull_encoding'] != 'None') &
        (summary_df['gaze_encoding'] == 'None') &
        (summary_df['partner_PC1_encoding'] == 'None')
    ].copy()
    significant_gaze_df = summary_df[
        (summary_df['pull_encoding'] == 'None') &
        (summary_df['gaze_encoding'] != 'None') &
        (summary_df['partner_PC1_encoding'] == 'None')
    ].copy()
    significant_partner_df = summary_df[
        (summary_df['pull_encoding'] == 'None') &
        (summary_df['gaze_encoding'] == 'None') &
        (summary_df['partner_PC1_encoding'] != 'None')
    ].copy()

    ax11 = plt.subplot(2, 3, 4)
    ax12 = plt.subplot(2, 3, 5)
    ax13 = plt.subplot(2, 3, 6)
    plot_encoding_type_venn(ax11, significant_pull_df, 'pull', 'Pull Encoding (in single-coded neurons)')
    plot_encoding_type_venn(ax12, significant_gaze_df, 'gaze', 'Gaze Encoding (in single-coded neurons)')
    plot_encoding_type_venn(ax13, significant_partner_df, 'partner_PC1', 'Partner PC1 Encoding (in single-coded neurons)')

    plt.tight_layout(rect=[0, 0, 1, 0.95])


    # --- 5. Bar plot: triple-encoded neurons also encoding force ---
    fig_force, ax_force = plt.subplots(figsize=(6, 5))
    fig_force.suptitle(f'Triple-Encoded Neurons\nalso encoding {force_to_look}',
                       fontsize=14, fontweight='bold')
    bars = ax_force.bar(
        [f'Triple +\n{force_to_look}', 'Triple only'],
        [n_triple_and_force, n_triple - n_triple_and_force],
        color=['crimson', 'gray']
    )
    for bar in bars:
        ax_force.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                      str(int(bar.get_height())), ha='center', va='bottom', fontsize=12)
    ax_force.set_ylabel('Number of Neurons', fontsize=12)
    ax_force.set_title(f'{n_triple_and_force}/{n_triple} ({percent_triple_and_force:.1f}%)', fontsize=12)
    plt.tight_layout()


    # --- 6. Bar Plots with Statistical Annotation ---
    import scipy.stats as stats

    def calculate_predictive_percentage(df, base_name):
        encoding_col = f'{base_name}_encoding'
        encoding_df = df[df[encoding_col] != 'None']
        if encoding_df.empty:
            return 0, 0, 0
        predictive_df = encoding_df[encoding_df[encoding_col].isin(['Predictive', 'Both'])]
        percentage = (len(predictive_df) / len(encoding_df)) * 100
        return percentage, len(predictive_df), len(encoding_df) - len(predictive_df)

    def calculate_dual_predictive_percentage(df):
        pool_df = df[(df['pull_encoding'] != 'None') | (df['gaze_encoding'] != 'None')]
        if pool_df.empty:
            return 0, 0, 0
        dual_predictive_df = pool_df[
            (pool_df['pull_encoding'].isin(['Predictive', 'Both'])) &
            (pool_df['gaze_encoding'].isin(['Predictive', 'Both']))
        ]
        percentage = (len(dual_predictive_df) / len(pool_df)) * 100
        return percentage, len(dual_predictive_df), len(pool_df) - len(dual_predictive_df)

    def perform_proportion_test(all_counts, triple_counts):
        table = [[triple_counts[0], triple_counts[1]], [all_counts[0], all_counts[1]]]
        _, p_value = stats.fisher_exact(table, alternative='greater')
        return p_value

    def format_p_value(p):
        if p < 0.001: return "p < 0.001"
        elif p < 0.01: return "p < 0.01"
        elif p < 0.05: return "p < 0.05"
        else: return f"p = {p:.2f}"

    def add_significance_annotation(ax, x1, x2, y, h, text):
        ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, c='black')
        ax.text((x1 + x2) * 0.5, y + h, text, ha='center', va='bottom', color='black', fontsize=14)

    fig3, (ax14, ax15, ax16) = plt.subplots(1, 3, figsize=(18, 6))
    fig3.suptitle(f'Proportion of Predictive Neurons for {recordedanimals[0]}', fontsize=20, fontweight='bold')
    bar_labels = ['All Encoding', 'Triple-Encoding']
    bar_colors = ['gray', 'crimson']

    percent_all_pull, pred_all_pull, not_pred_all_pull = calculate_predictive_percentage(summary_df, 'pull')
    percent_triple_pull, pred_triple_pull, not_pred_triple_pull = calculate_predictive_percentage(significant_all_three_df, 'pull')
    p_pull = perform_proportion_test((pred_all_pull, not_pred_all_pull), (pred_triple_pull, not_pred_triple_pull))
    ax14.bar(bar_labels, [percent_all_pull, percent_triple_pull], color=bar_colors)
    ax14.set_title('Pull Encoding', fontsize=16)
    ax14.set_ylabel('Percentage of Neurons (%)', fontsize=14)
    ax14.set_ylim(0, 100)
    add_significance_annotation(ax14, 0, 1, max(percent_all_pull, percent_triple_pull) + 5, 2, format_p_value(p_pull))

    percent_all_gaze, pred_all_gaze, not_pred_all_gaze = calculate_predictive_percentage(summary_df, 'gaze')
    percent_triple_gaze, pred_triple_gaze, not_pred_triple_gaze = calculate_predictive_percentage(significant_all_three_df, 'gaze')
    p_gaze = perform_proportion_test((pred_all_gaze, not_pred_all_gaze), (pred_triple_gaze, not_pred_triple_gaze))
    ax15.bar(bar_labels, [percent_all_gaze, percent_triple_gaze], color=bar_colors)
    ax15.set_title('Gaze Encoding', fontsize=16)
    ax15.set_ylim(0, 100)
    add_significance_annotation(ax15, 0, 1, max(percent_all_gaze, percent_triple_gaze) + 5, 2, format_p_value(p_gaze))

    percent_all_dual, pred_all_dual, not_pred_all_dual = calculate_dual_predictive_percentage(summary_df)
    percent_triple_dual, pred_triple_dual, not_pred_triple_dual = calculate_dual_predictive_percentage(significant_all_three_df)
    p_dual = perform_proportion_test((pred_all_dual, not_pred_all_dual), (pred_triple_dual, not_pred_triple_dual))
    ax16.bar(bar_labels, [percent_all_dual, percent_triple_dual], color=bar_colors)
    ax16.set_title('Dual Predictive (Pull & Gaze)', fontsize=16)
    ax16.set_ylim(0, 100)
    add_significance_annotation(ax16, 0, 1, max(percent_all_dual, percent_triple_dual) + 5, 2, format_p_value(p_dual))

    fig3.tight_layout(rect=[0, 0, 1, 0.95])


    # --- 7. Force Encoding Overlap Venn Diagrams ---
    self_force_set = get_any_encoding_set(summary_df, 'self_deltaforce')
    other_force_set = get_any_encoding_set(summary_df, 'other_deltaforce')

    fig4, (ax17, ax18) = plt.subplots(1, 2, figsize=(16, 7))
    fig4.suptitle('Force Encoding Overlap', fontsize=22, fontweight='bold')

    none_pull_partner_selfforce = summary_df[
        (summary_df['pull_encoding'] == 'None') &
        (summary_df['partner_PC1_encoding'] == 'None') &
        (summary_df['self_deltaforce_encoding'] == 'None')
    ].shape[0]
    venn3([pull_set, partner_set, self_force_set],
          set_labels=('Pull', 'Partner PC1', 'Self Force'), ax=ax17)
    ax17.set_title('Pull vs. Partner PC1 vs. Self Force', fontsize=16)
    ax17.text(0.95, 0.05, f"None in all three: {none_pull_partner_selfforce}",
              transform=ax17.transAxes, ha='right', va='bottom',
              fontsize=12, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))

    none_pull_partner_otherforce = summary_df[
        (summary_df['pull_encoding'] == 'None') &
        (summary_df['partner_PC1_encoding'] == 'None') &
        (summary_df['other_deltaforce_encoding'] == 'None')
    ].shape[0]
    venn3([pull_set, partner_set, other_force_set],
          set_labels=('Pull', 'Partner PC1', 'Other Force'), ax=ax18)
    ax18.set_title('Pull vs. Partner PC1 vs. Other Force', fontsize=16)
    ax18.text(0.95, 0.05, f"None in all three: {none_pull_partner_otherforce}",
              transform=ax18.transAxes, ha='right', va='bottom',
              fontsize=12, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))

    plt.tight_layout(rect=[0, 0, 1, 0.95])


    # --- Save all figures ---
    savefig = 1
    if savefig:
        figsavefolder = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents_neuralGLMfitting_BasisKernelsForContVaris_singlecam/"+\
                        cameraID+"/"+savefile_sufix+"/"+recordedanimals[0]+"_neuralGLM/"
        if not os.path.exists(figsavefolder):
            os.makedirs(figsavefolder)

        fig.savefig(figsavefolder+recordedanimals[0]+
                    '_NeuronEncoding_PullOrGaze_and_partnerAction_VennGram_LOOmethods_'+
                    savefile_sufix+'_'+cond_toplot_type+'.pdf')
        fig2.savefig(figsavefolder+recordedanimals[0]+
                     '_NeuronEncoding_TriEncodedNeurons_PredictiveReactive_VennGram_LOOmethods_'+
                     savefile_sufix+'_'+cond_toplot_type+'.pdf')
        fig3.savefig(figsavefolder+recordedanimals[0]+
                     '_NeuronEncoding_TriEncodedNeurons_PredictiveProportion_BarPlot_LOOmethods_'+
                     savefile_sufix+'_'+cond_toplot_type+'.pdf')
        fig4.savefig(figsavefolder+recordedanimals[0]+
                     '_NeuronEncoding_ForceEncoding_VennGram_LOOmethods_'+
                     savefile_sufix+'_'+cond_toplot_type+'.pdf')
        fig_force.savefig(figsavefolder+recordedanimals[0]+
                          f'_NeuronEncoding_TripleAndForce_{force_to_look}_LOOmethods_'+
                          savefile_sufix+'_'+cond_toplot_type+'.pdf')

In [ ]:
# get the statistics of the tri-encoding neurons and the tri-encoding + deltaforce encoding
if 1:
    from scipy.stats import hypergeom, fisher_exact, binomtest

    # --- Test 1: Is triple-encoding significant? ---
    n_total = len(summary_df)

    n_pull = len(pull_set)
    n_gaze = len(gaze_set)
    n_partner = len(partner_set)

    # individual encoding rates
    p_pull = n_pull / n_total
    p_gaze = n_gaze / n_total
    p_partner = n_partner / n_total

    # expected triple-encoding rate if independent
    p_triple_expected = p_pull * p_gaze * p_partner
    n_triple_expected = p_triple_expected * n_total

    # binomial test: is observed triple count > expected?
    binom_result = binomtest(n_triple, n_total, p_triple_expected, alternative='greater')

    print("=== Test 1: Is triple-encoding significant? ===")
    print(f"Total neurons: {n_total}")
    print(f"P(pull): {p_pull:.3f}, P(gaze): {p_gaze:.3f}, P(partner PC1): {p_partner:.3f}")
    print(f"Expected triple-encoding by chance: {n_triple_expected:.1f}")
    print(f"Observed triple-encoding: {n_triple}")
    print(f"Binomial test p-value: {binom_result.pvalue:.4f}")

    # --- Test 2: Is overlap between triple-encoding and force significant? ---
    n_force = len(force_set)
    n_overlap = len(triple_set & force_set)

    contingency = [
        [n_overlap,           n_triple - n_overlap],
        [n_force - n_overlap, n_total - n_triple - n_force + n_overlap]
    ]
    odds_ratio, p_fisher = fisher_exact(contingency, alternative='greater')
    p_hypergeom = hypergeom.sf(n_overlap - 1, n_total, n_triple, n_force)

    print(f"\n=== Test 2: Is triple-encoding & {force_to_look} overlap significant? ===")
    print(f"Triple-encoded: {n_triple}")
    print(f"Force-encoding: {n_force}")
    print(f"Observed overlap: {n_overlap}")
    print(f"Expected overlap by chance: {n_triple * n_force / n_total:.1f}")
    print(f"Fisher's exact test p-value: {p_fisher:.4f}, odds ratio: {odds_ratio:.2f}")
    print(f"Hypergeometric test p-value: {p_hypergeom:.4f}")

In [ ]:
# get the firing rate aligned at pull for target neurons

# great example： kanga 20241010_56, 20241010_196

if 1:
    # load saved data
    data_saved_subfolder_2 = data_saved_folder+'data_saved_singlecam_wholebody'+savefile_sufix+'/'+cameraID+'/'+ \
                animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+recordedanimals[0]+'Recorded'+'/'

    with open(data_saved_subfolder_2+'/bhvevents_aligned_FR_allevents_all_dates_'+animal1_fixedorders[0]+animal2_fixedorders[0]+'.pkl', 'rb') as f:
        bhvevents_aligned_FR_allevents_all_dates = pickle.load(f) 
                
    
    # --- Choose which force variable to look at ---
    # force_to_look = 'self_deltaforce'  # change to 'other_deltaforce' for other force
    # force_to_look = 'other_deltaforce'
    if cond_toplot_type == 'other_EffortBasedMC':
        force_to_look = 'other_deltaforce'
    else:
        force_to_look = 'self_deltaforce'

    # --- Find neurons significant in pull, partner PC1, AND the chosen force ---
    triple_encoded_force_df = summary_df[
        (summary_df['pull_encoding'] != 'None') &
        (summary_df['partner_PC1_encoding'] != 'None') &
        (summary_df[f'{force_to_look}_encoding'] != 'None')
    ].copy()
    
    # --- Pick example neuron with constraints ---

    # constraint 1: triple encoded + pull predictive neuron (Predictive or Both)
    predictive_pull_df = triple_encoded_force_df[
        triple_encoded_force_df['pull_encoding'].isin(['Predictive', 'Both', 'Reactive'])
    ].copy()

    print(f"After pull predictive filter: {len(predictive_pull_df)} neurons")

    # constraint 2 & 3: firing peak > 0 and all blocks present
    valid_neurons = []

    for _, row in predictive_pull_df.iterrows():
        unique_neuron_str_tmp = row['unique_neuron']
        last_underscore_idx = unique_neuron_str_tmp.rfind('_')
        date_tmp = unique_neuron_str_tmp[:last_underscore_idx]
        neuronID_tmp = int(unique_neuron_str_tmp[last_underscore_idx+1:])

        # check date exists
        if date_tmp not in bhvevents_aligned_FR_allevents_all_dates:
            continue

        block_keys_tmp = list(bhvevents_aligned_FR_allevents_all_dates[date_tmp].keys())

        # find non-None block to get animal names
        non_empty_block_tmp = None
        for key in block_keys_tmp:
            if bhvevents_aligned_FR_allevents_all_dates[date_tmp][key] is not None:
                non_empty_block_tmp = key
                break
        if non_empty_block_tmp is None:
            continue

        event_keys_tmp = list(bhvevents_aligned_FR_allevents_all_dates[date_tmp][non_empty_block_tmp].keys())
        animal_names_tmp = list(set([k.split(' ')[0] for k in event_keys_tmp]))
        try:
            self_animal_tmp = [a for a in animal_names_tmp if a in recordedanimals[0].lower()][0]
        except IndexError:
            continue
        self_pull_key_tmp = self_animal_tmp + ' pull'

        # constraint 3: ALL blocks must be non-None and have data for this neuron
        all_blocks_valid = True
        all_mean_FR = []
        for key in block_keys_tmp:
            # fail if any block is None
            if bhvevents_aligned_FR_allevents_all_dates[date_tmp][key] is None:
                all_blocks_valid = False
                break
            try:
                data = bhvevents_aligned_FR_allevents_all_dates[date_tmp][key][self_pull_key_tmp][str(neuronID_tmp)]['FR_allevents']
                if data.shape[1] == 0:
                    all_blocks_valid = False
                    break
                all_mean_FR.append(np.nanmean(data, axis=1))
            except KeyError:
                all_blocks_valid = False
                break

        if not all_blocks_valid or len(all_mean_FR) == 0:
            continue

        # constraint 2: firing peak > 0 (peak of mean across all blocks)
        overall_mean = np.nanmean(np.vstack(all_mean_FR), axis=0)
        if np.nanmax(overall_mean) <= 0:
            continue

        valid_neurons.append(row)

    print(f"After all constraints: {len(valid_neurons)} neurons")

    if len(valid_neurons) == 0:
        print("No neurons passed all constraints!")
    else:
        # randomly pick one from valid neurons
        example_row = random.choice(valid_neurons)
        unique_neuron_str = example_row['unique_neuron']
        last_underscore_idx = unique_neuron_str.rfind('_')
        example_date = unique_neuron_str[:last_underscore_idx]
        example_neuronID = int(unique_neuron_str[last_underscore_idx+1:])

        print(f"\nSelected neuron: {unique_neuron_str}")
        print(f"Date: {example_date}")
        print(f"Neuron ID: {example_neuronID}")
        print(f"Encoding: {example_row['pull_encoding']} pull, {example_row['partner_PC1_encoding']} partner PC1, {example_row[f'{force_to_look}_encoding']} {force_to_look}")
    

    # --- Extract block info and calculate delta force ---
    block_keys = list(bhvevents_aligned_FR_allevents_all_dates[example_date].keys())
    print(f"Block keys: {block_keys}")

    # Parse lever1 and lever2 force from each block key
    block_forces = []
    for key in block_keys:
        lever1_force, lever2_force = key.split('&')
        block_forces.append({
            'key': key,
            'lever1_force': int(lever1_force),
            'lever2_force': int(lever2_force)
        })

    # Figure out which lever changed across blocks
    lever1_values = [b['lever1_force'] for b in block_forces]
    lever2_values = [b['lever2_force'] for b in block_forces]

    lever1_changed = len(set(lever1_values)) > 1
    lever2_changed = len(set(lever2_values)) > 1

    if lever1_changed:
        force_col = 'lever1_force'
    elif lever2_changed:
        force_col = 'lever2_force'
    else:
        force_col = 'lever1_force'  # fallback if nothing changed

    print(f"Changing force: {force_col}")

    # Calculate delta force relative to previous block, first block = 0
    for i, block in enumerate(block_forces):
        if i == 0:
            block['delta_force'] = 0
        else:
            block['delta_force'] = block[force_col] - block_forces[i-1][force_col]

    print("\nBlock info:")
    for block in block_forces:
        print(f"  {block['key']} -> {force_col}={block[force_col]}, delta_force={block['delta_force']}")

    # --- Infer animal names from event keys of first non-None block ---
    non_empty_block = None
    for block in block_forces:
        key = block['key']
        if bhvevents_aligned_FR_allevents_all_dates[example_date][key] is not None:
            non_empty_block = key
            break

    if non_empty_block is None:
        print("ERROR: all blocks are None!")
    else:
        print(f"\nNon-empty block found: {non_empty_block}")
        event_keys = list(bhvevents_aligned_FR_allevents_all_dates[example_date][non_empty_block].keys())
        print(f"Event keys: {event_keys}")

        # extract unique animal names from keys like 'kanga pull', 'dodson pull'
        animal_names = list(set([k.split(' ')[0] for k in event_keys]))
        print(f"Animal names found: {animal_names}")

        # assign self and partner
        self_animal = [a for a in animal_names if a in recordedanimals[0].lower()][0]
        partner_animal = [a for a in animal_names if a not in recordedanimals[0].lower()][0]

        self_pull_key = self_animal + ' pull'
        partner_pull_key = partner_animal + ' pull'

        print(f"Self pull key: {self_pull_key}")
        print(f"Partner pull key: {partner_pull_key}")

        # --- Filter out None and empty blocks ---
        valid_block_forces = []
        for block in block_forces:
            key = block['key']
            if bhvevents_aligned_FR_allevents_all_dates[example_date][key] is None:
                print(f"Skipping None block: {key}")
                continue
            try:
                data = bhvevents_aligned_FR_allevents_all_dates[example_date][key][self_pull_key][str(example_neuronID)]['FR_allevents']
                if data.shape[1] > 0:
                    valid_block_forces.append(block)
            except KeyError:
                print(f"Skipping empty block: {key}")

        print(f"\nValid blocks: {[b['key'] for b in valid_block_forces]}")
        
        

        # --- Plot self pull and partner pull side by side ---
        t = np.linspace(-4, 4, 240)
        cmap = plt.cm.coolwarm
        delta_forces = [b['delta_force'] for b in valid_block_forces]
        max_abs_delta = max(abs(d) for d in delta_forces) if any(d != 0 for d in delta_forces) else 1
        norm = plt.Normalize(-max_abs_delta, max_abs_delta)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        fig.suptitle(f'Example Neuron: {unique_neuron_str}\nSelf vs. Partner Pull-aligned FR by block',
                     fontsize=14, fontweight='bold')

        for ax, pull_key, title in zip([ax1, ax2],
                                        [self_pull_key, partner_pull_key],
                                        ['Self Pull', 'Partner Pull']):
            for block in valid_block_forces:
                key = block['key']
                delta = block['delta_force']
                color = cmap(norm(delta))

                try:
                    FR_allevents = bhvevents_aligned_FR_allevents_all_dates[example_date][key][pull_key][str(example_neuronID)]['FR_allevents']
                    mean_FR = np.nanmean(FR_allevents, axis=1)
                    sem_FR = np.nanstd(FR_allevents, axis=1) / np.sqrt(FR_allevents.shape[1])

                    label = f'{key} (Δ={delta:+d}, n={FR_allevents.shape[1]})'
                    ax.plot(t, mean_FR, color=color, linewidth=2, label=label)
                    ax.fill_between(t, mean_FR - sem_FR, mean_FR + sem_FR, alpha=0.2, color=color)
                except KeyError:
                    print(f"No data for block {key}, event {pull_key}")

            ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Pull onset')
            ax.set_xlabel('Time (s)', fontsize=12)
            ax.set_title(title, fontsize=12)
            ax.legend(fontsize=9, loc='upper right')

        ax1.set_ylabel('Firing Rate (Hz)', fontsize=12)

        plt.tight_layout()
    
    
        # --- Add two more panels: mean firing vs delta force (per trial) ---
        t_pre = np.linspace(-4, 4, 240)
        t_pre_mask = t_pre < 0  # -4 to 0s window for self pull

        fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(12, 5))
        fig2.suptitle(f'Example Neuron: {unique_neuron_str}\nMean Firing Rate vs Delta Force',
                      fontsize=14, fontweight='bold')

        # collect per-trial mean FR per block
        delta_force_list = []
        pertrial_FR_selfpull_list = []
        pertrial_FR_partnerpull_list = []

        for block in valid_block_forces:
            key = block['key']
            delta = block['delta_force']

            # self pull: mean over -4 to 0s per trial
            try:
                FR_self = bhvevents_aligned_FR_allevents_all_dates[example_date][key][self_pull_key][str(example_neuronID)]['FR_allevents']
                # FR_self shape: (240, n_trials)
                pertrial_mean_pre = np.nanmean(FR_self[t_pre_mask, :], axis=0)  # (n_trials,)
            except KeyError:
                pertrial_mean_pre = np.array([np.nan])

            # partner pull: mean over -4 to 4s per trial
            try:
                FR_partner = bhvevents_aligned_FR_allevents_all_dates[example_date][key][partner_pull_key][str(example_neuronID)]['FR_allevents']
                pertrial_mean_partner = np.nanmean(FR_partner, axis=0)  # (n_trials,)
            except KeyError:
                pertrial_mean_partner = np.array([np.nan])

            delta_force_list.append(delta)
            pertrial_FR_selfpull_list.append(pertrial_mean_pre)
            pertrial_FR_partnerpull_list.append(pertrial_mean_partner)

        # sort by delta force for cleaner x-axis
        sort_idx = np.argsort(delta_force_list)
        delta_force_sorted = np.array(delta_force_list)[sort_idx]
        selfpull_sorted = [pertrial_FR_selfpull_list[i] for i in sort_idx]
        partnerpull_sorted = [pertrial_FR_partnerpull_list[i] for i in sort_idx]
        keys_sorted = [valid_block_forces[i]['key'] for i in sort_idx]
        colors_sorted = [cmap(norm(d)) for d in delta_force_sorted]
        
        from scipy.stats import f_oneway

        # ANOVA across blocks for self pull
        self_groups = [data[~np.isnan(data)] for data in selfpull_sorted]
        if all(len(g) > 1 for g in self_groups):
            f_self, p_self = f_oneway(*self_groups)
        else:
            f_self, p_self = np.nan, np.nan

        # ANOVA across blocks for partner pull
        partner_groups = [data[~np.isnan(data)] for data in partnerpull_sorted]
        if all(len(g) > 1 for g in partner_groups):
            f_partner, p_partner = f_oneway(*partner_groups)
        else:
            f_partner, p_partner = np.nan, np.nan

        print(f"Self pull ANOVA: F={f_self:.3f}, p={p_self:.4f}")
        print(f"Partner pull ANOVA: F={f_partner:.3f}, p={p_partner:.4f}")

        for ax, data_list, title, p_val in zip(
            [ax3, ax4],
            [selfpull_sorted, partnerpull_sorted],
            ['Self Pull: Mean FR (-4 to 0s) vs Delta Force',
             'Partner Pull: Mean FR (-4 to 4s) vs Delta Force'],
            [p_self, p_partner]
        ):
            positions = np.arange(len(delta_force_sorted))

            for pos, data, color, key, delta in zip(positions, data_list, colors_sorted, keys_sorted, delta_force_sorted):
                # violin
                parts = ax.violinplot(data[~np.isnan(data)], positions=[pos], widths=0.6, 
                                      showmeans=True, showextrema=False)
                for pc in parts['bodies']:
                    pc.set_facecolor(color)
                    pc.set_alpha(0.6)
                parts['cmeans'].set_color('black')
                parts['cmeans'].set_linewidth(2)

            ax.set_xticks(positions)
            ax.set_xticklabels([f'{k}\nΔ={d:+d}' for k, d in zip(keys_sorted, delta_force_sorted)], fontsize=9)
            ax.axhline(0, color='gray', linestyle='--', linewidth=1)
            ax.set_ylabel('Mean Firing Rate (Hz)', fontsize=12)
            ax.set_title(title, fontsize=11)
            
            # add ANOVA annotation
            p_str = f"p < 0.001" if p_val < 0.001 else f"p < 0.01" if p_val < 0.01 else f"p < 0.05" if p_val < 0.05 else f"p = {p_val:.2f}"
            ax.text(0.98, 0.98, f"One-way ANOVA: {p_str}",
                    transform=ax.transAxes, ha='right', va='top',
                    fontsize=10, bbox=dict(boxstyle='round,pad=0.3', fc='wheat', alpha=0.5))


        plt.tight_layout()
        
        
        # --- Load bootstrap GLM temporal filters for example date ---
        current_dir = data_saved_folder+'/bhv_events_singlecam_wholebody_with_neuralglm_model'+savefile_sufix+'/'+\
                      animal1_fixedorders[0]+animal2_fixedorders[0]+'_'+recordedanimals[0]+'Recorded'
        add_date_dir = os.path.join(current_dir, cameraID+'/'+example_date)


        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_tempFilter.pkl', 'rb') as f:
            neuralGLM_mainAxes_partnerPC1_kernels_tempFilter = pickle.load(f)
        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_kernels_tempFilter_shf.pkl', 'rb') as f:
            neuralGLM_mainAxes_partnerPC1_kernels_tempFilter_shf = pickle.load(f)

        # --- Plot temporal filters for example neuron ---
        var_toglm_names_mainAxes_partnerPC1 = ['var_pull', 'var_gaze', 'var_juice', 'partner_PC1']
        t_filter = np.linspace(-4, 4, 240)

        # get filters for example neuron: shape (100, 4, 240)
        filters_real = neuralGLM_mainAxes_partnerPC1_kernels_tempFilter[example_neuronID]
        filters_shf = neuralGLM_mainAxes_partnerPC1_kernels_tempFilter_shf[example_neuronID]

        fig3, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
        fig3.suptitle(f'Temporal Filters: {unique_neuron_str}', fontsize=14, fontweight='bold')

        for ax, var_idx, var_name in zip(axes, range(4), var_toglm_names_mainAxes_partnerPC1):
            # real data: mean and std across bootstraps
            mean_real = np.mean(filters_real[:, var_idx, :], axis=0)
            std_real = np.std(filters_real[:, var_idx, :], axis=0)

            # shuffled data: mean and std across bootstraps
            mean_shf = np.mean(filters_shf[:, var_idx, :], axis=0)
            std_shf = np.std(filters_shf[:, var_idx, :], axis=0)

            # plot shuffled first (background)
            ax.plot(t_filter, mean_shf, color='gray', linewidth=1.5, label='Shuffled', alpha=0.7)
            ax.fill_between(t_filter, mean_shf - std_shf, mean_shf + std_shf, alpha=0.2, color='gray')

            # plot real on top
            ax.plot(t_filter, mean_real, color='steelblue', linewidth=2, label='Real')
            ax.fill_between(t_filter, mean_real - std_real, mean_real + std_real, alpha=0.3, color='steelblue')

            ax.axvline(0, color='black', linestyle='--', linewidth=1, label='spike time')
            ax.axhline(0, color='gray', linestyle=':', linewidth=1)
            ax.set_title(var_name, fontsize=12)
            ax.set_xlabel('Time (s)', fontsize=10)
            if ax == axes[0]:
                ax.set_ylabel('Filter weight (a.u.)', fontsize=10)
            ax.legend(fontsize=8)

        plt.tight_layout()
        
        
       # --- Load bootstrap GLM scalar coefficients for example date ---

        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_scalar_coef.pkl', 'rb') as f:
            neuralGLM_mainAxes_partnerPC1_scalar_coef = pickle.load(f)
        with open(add_date_dir+'/neuralGLM_mainAxes_partnerPC1_scalar_coef_shf.pkl', 'rb') as f:
            neuralGLM_mainAxes_partnerPC1_scalar_coef_shf = pickle.load(f)

        # scalar coef shape: (100, 2) — 100 bootstraps, 2 scalars (self, other)
        scalar_real = neuralGLM_mainAxes_partnerPC1_scalar_coef[example_neuronID]   # (100, 2)
        scalar_shf = neuralGLM_mainAxes_partnerPC1_scalar_coef_shf[example_neuronID] # (100, 2)

        scalar_names = ['self_deltaforce', 'other_deltaforce']

        fig4, axes_scalar = plt.subplots(1, 2, figsize=(10, 4))
        fig4.suptitle(f'Scalar Coefficients: {unique_neuron_str}', fontsize=14, fontweight='bold')

        for ax, scalar_idx, scalar_name in zip(axes_scalar, range(2), scalar_names):
            real_coefs = scalar_real[:, scalar_idx]   # (100,)
            shf_coefs = scalar_shf[:, scalar_idx]     # (100,)

            # violin for real and shuffled
            parts = ax.violinplot([real_coefs, shf_coefs], positions=[0, 1],
                                  widths=0.6, showmeans=True, showextrema=False)
            parts['bodies'][0].set_facecolor('steelblue')
            parts['bodies'][0].set_alpha(0.6)
            parts['bodies'][1].set_facecolor('gray')
            parts['bodies'][1].set_alpha(0.6)
            parts['cmeans'].set_color('black')
            parts['cmeans'].set_linewidth(2)

            # add individual points
            ax.scatter(np.zeros(len(real_coefs)) + np.random.normal(0, 0.02, len(real_coefs)),
                       real_coefs, color='steelblue', alpha=0.3, s=10)
            ax.scatter(np.ones(len(shf_coefs)) + np.random.normal(0, 0.02, len(shf_coefs)),
                       shf_coefs, color='gray', alpha=0.3, s=10)

            # test real vs shuffled
            from scipy.stats import mannwhitneyu
            _, p_val = mannwhitneyu(real_coefs, shf_coefs, alternative='two-sided')
            p_str = f"p < 0.001" if p_val < 0.001 else f"p < 0.01" if p_val < 0.01 else f"p < 0.05" if p_val < 0.05 else f"p = {p_val:.2f}"

            ax.axhline(0, color='gray', linestyle='--', linewidth=1)
            ax.set_xticks([0, 1])
            ax.set_xticklabels(['Real', 'Shuffled'], fontsize=11)
            ax.set_title(f'{scalar_name}\n{p_str}', fontsize=12)
            ax.set_ylabel('Scalar Coefficient', fontsize=11)

        plt.tight_layout()
        
        # --- Save all figures ---
        savefig = 1
        if savefig:
            figsavefolder = data_saved_folder+"fig_for_basic_neural_analysis_allsessions_basicEvents_neuralGLMfitting_BasisKernelsForContVaris_singlecam/"+\
                            cameraID+"/"+savefile_sufix+"/"+recordedanimals[0]+"_neuralGLM/example_neuron/"
            if not os.path.exists(figsavefolder):
                os.makedirs(figsavefolder)

            fig.savefig(figsavefolder+unique_neuron_str+
                        '_SelfPartnerPull_FR_byBlock_'+
                        savefile_sufix+'_'+cond_toplot_type+'.pdf')

            fig2.savefig(figsavefolder+unique_neuron_str+
                         '_MeanFR_vs_DeltaForce_Violin_'+
                         savefile_sufix+'_'+cond_toplot_type+'.pdf')

            fig3.savefig(figsavefolder+unique_neuron_str+
                         '_TemporalFilters_RealVsShuffled_'+
                         savefile_sufix+'_'+cond_toplot_type+'.pdf')

            fig4.savefig(figsavefolder+unique_neuron_str+
                         '_ScalarCoefs_DeltaForce_RealVsShuffled_'+
                         savefile_sufix+'_'+cond_toplot_type+'.pdf')
        

In [ ]:
bhvevents_aligned_FR_allevents_all_dates[example_date]

In [ ]:
np.shape(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter_summary[example_date][example_neuronID]['var_pull_past'])





In [ ]:
plt.plot(neuralGLM_mainAxes_partnerPC1_LOOmethod_kernels_tempFilter[5]['var_pull_past'])

In [ ]:
np.shape(neuralGLM_mainAxes_partnerPC1_kernels_tempFilter[5])

In [ ]:
neuralGLM_mainAxes_partnerPC1_kernels_tempFilter.keys()